# 15 - Variational Quantum Eigensolver (VQE)

Find the ground-state energy of a simple Hamiltonian using a hybrid quantum-classical loop.

**Concepts:** Ansatz, expectation values, parameter optimization

In [ ]:
import quantsdk as qs
import math
import numpy as np

## Simple H2-like Hamiltonian

We approximate the ground state of $H = -Z_0 Z_1 + 0.5 X_0 + 0.5 X_1$
using a parameterized ansatz.

In [ ]:
def ansatz(theta: float) -> qs.Circuit:
    """Simple 2-qubit ansatz with one parameter."""
    circuit = qs.Circuit(2, name=f"ansatz-{theta:.3f}")
    circuit.ry(0, theta)
    circuit.cx(0, 1)
    return circuit

def measure_zz(theta: float, shots: int = 4000) -> float:
    """Measure <Z0 Z1> for the ansatz."""
    circuit = ansatz(theta)
    circuit.measure_all()
    result = qs.run(circuit, shots=shots, seed=42)
    # Z eigenvalues: |0> -> +1, |1> -> -1
    exp = 0.0
    for bs, count in result.counts.items():
        z0 = 1 - 2 * int(bs[0])
        z1 = 1 - 2 * int(bs[1])
        exp += z0 * z1 * count
    return exp / shots

def measure_x(theta: float, qubit: int, shots: int = 4000) -> float:
    """Measure <X> on a given qubit by rotating to X basis."""
    circuit = ansatz(theta)
    circuit.h(qubit)  # Rotate to X basis
    circuit.measure(qubit)
    result = qs.run(circuit, shots=shots, seed=42)
    exp = 0.0
    for bs, count in result.counts.items():
        val = 1 - 2 * int(bs[qubit])
        exp += val * count
    return exp / shots

In [ ]:
def energy(theta: float) -> float:
    """Compute <H> = -<ZZ> + 0.5*<X0> + 0.5*<X1>."""
    zz = measure_zz(theta)
    x0 = measure_x(theta, 0)
    x1 = measure_x(theta, 1)
    return -zz + 0.5 * x0 + 0.5 * x1

# Parameter sweep
thetas = np.linspace(0, 2 * math.pi, 20)
energies = [energy(t) for t in thetas]

best_idx = int(np.argmin(energies))
print(f"Best theta: {thetas[best_idx]:.3f} rad")
print(f"Best energy: {energies[best_idx]:.3f}")
print(f"\nEnergy landscape (sample):")
for i in range(0, len(thetas), 4):
    bar = '#' * int((energies[i] + 2) * 10)
    print(f"  theta={thetas[i]:5.2f}: E={energies[i]:+.3f} {bar}")

## Simple Gradient-Free Optimization

In [ ]:
# Simple grid refinement optimizer
best_theta = thetas[best_idx]
for resolution in [0.5, 0.1, 0.01]:
    search = np.linspace(best_theta - resolution * 5, best_theta + resolution * 5, 11)
    search_energies = [energy(t) for t in search]
    best_theta = search[int(np.argmin(search_energies))]

final_energy = energy(best_theta)
print(f"Optimized theta: {best_theta:.4f} rad")
print(f"Final energy: {final_energy:.4f}")